# Notebook 03 · Feature Engineering

## Objetivo

Construir variables derivadas que enriquezcan el dataset analítico y faciliten el análisis y la visualización en Power BI.

Las variables creadas en este notebook no sustituyen los análisis realizados previamente en SQL ni en el Notebook 02. Su objetivo es complementar el modelo analítico mediante nuevas variables orientadas al negocio, facilitando la segmentación de clientes y simplificando el desarrollo del dashboard.

---

## Resultado esperado

Al finalizar este notebook se obtendrá un nuevo dataset denominado `customer_features_df`, que incorporará las variables derivadas aprobadas durante la fase de diseño y servirá como base para la preparación del modelo final destinado a Power BI.

In [27]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# =============================================================================

from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [28]:
# =============================================================================
# CONFIGURACIÓN DEL ENTORNO
# =============================================================================

# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Ruta de la base de datos SQLite
DATABASE_PATH = PROJECT_ROOT / "data" / "raw" / "database" / "bank_sqlite.db"

# Conexión a la base de datos
try:
    connection = sqlite3.connect(DATABASE_PATH)
    print("✅ Conexión con la base de datos establecida correctamente.")
except sqlite3.Error as error:
    print(f"❌ Error al conectar con la base de datos: {error}")
    raise

✅ Conexión con la base de datos establecida correctamente.


# 4. Carga del dataset analítico

En esta sección se reconstruye el dataset analítico generado durante el Notebook 02.

El objetivo es garantizar que este notebook pueda ejecutarse de forma completamente independiente, utilizando como punto de partida el mismo conjunto de datos analíticos.

In [29]:
# =============================================================================
# CONSULTA DEL DATASET ANALÍTICO
# =============================================================================
# Se reutiliza la misma consulta SQL definida en el Notebook 02 para garantizar
# que ambos notebooks trabajen sobre el mismo dataset analítico.
# =============================================================================

query_customer_analysis = """
WITH

accounts_summary AS (

    SELECT
        customer_id,
        COUNT(account_id) AS total_accounts,
        SUM(balance_usd) AS total_balance
    FROM accounts
    GROUP BY customer_id

),

cards_summary AS (

    SELECT
        a.customer_id,
        COUNT(c.card_id) AS total_cards
    FROM accounts AS a
    INNER JOIN cards AS c
        ON a.account_id = c.account_id
    GROUP BY a.customer_id

),

loans_summary AS (

    SELECT
        customer_id,
        COUNT(loan_id) AS total_loans,
        SUM(loan_amount) AS total_loan_amount
    FROM loans
    GROUP BY customer_id

),

transactions_summary AS (

    SELECT
        a.customer_id,
        COUNT(t.transaction_id) AS total_transactions,
        SUM(t.amount_usd) AS total_transaction_amount
    FROM accounts AS a
    INNER JOIN transactions AS t
        ON a.account_id = t.account_id
    GROUP BY a.customer_id

)

SELECT
    c.customer_id,
    c.credit_score,

    COALESCE(a.total_accounts, 0) AS total_accounts,
    COALESCE(a.total_balance, 0) AS total_balance,

    COALESCE(cd.total_cards, 0) AS total_cards,

    COALESCE(l.total_loans, 0) AS total_loans,
    COALESCE(l.total_loan_amount, 0) AS total_loan_amount,

    COALESCE(t.total_transactions, 0) AS total_transactions,
    COALESCE(t.total_transaction_amount, 0) AS total_transaction_amount

FROM customers AS c

LEFT JOIN accounts_summary AS a
    ON c.customer_id = a.customer_id

LEFT JOIN cards_summary AS cd
    ON c.customer_id = cd.customer_id

LEFT JOIN loans_summary AS l
    ON c.customer_id = l.customer_id

LEFT JOIN transactions_summary AS t
    ON c.customer_id = t.customer_id;
"""

In [30]:
# =============================================================================
# CARGA DEL DATASET ANALÍTICO
# =============================================================================

try:
    customer_analysis_df = pd.read_sql(
        sql=query_customer_analysis,
        con=connection
    )

    print("✅ Dataset analítico cargado correctamente.")

except Exception as error:
    print(f"❌ Error al cargar el dataset analítico: {error}")
    raise

✅ Dataset analítico cargado correctamente.


# 5. Creación de variables derivadas

En esta sección se construyen las variables derivadas aprobadas durante la fase de diseño del proyecto.

Cada variable ha sido seleccionada por su capacidad para enriquecer el dataset analítico sin duplicar transformaciones realizadas previamente. Estas nuevas variables facilitarán la segmentación de clientes y simplificarán el desarrollo del dashboard en Power BI.

## 5.1 Total Products

### Descripción

Calcula el número total de productos financieros contratados por cada cliente, considerando cuentas bancarias, tarjetas y préstamos.

### Valor para el negocio

Representa el nivel de vinculación del cliente con la entidad mediante un único indicador cuantitativo. Esta variable facilitará la segmentación de clientes y servirá como base para nuevas métricas y visualizaciones en Power BI.

In [31]:
# =============================================================================
# CREACIÓN DEL DATASET ENRIQUECIDO
# =============================================================================

customer_features_df = customer_analysis_df.copy()

In [32]:
# =============================================================================
# FV-001 · TOTAL PRODUCTS
# =============================================================================

customer_features_df["total_products"] = (
    customer_features_df["total_accounts"]
    + customer_features_df["total_cards"]
    + customer_features_df["total_loans"]
)

In [33]:
# =============================================================================
# VERIFICACIÓN · TOTAL PRODUCTS
# =============================================================================

print(f"Valores nulos: {customer_features_df['total_products'].isna().sum()}")

display(
    customer_features_df[
        [
            "total_accounts",
            "total_cards",
            "total_loans",
            "total_products"
        ]
    ].head()
)

customer_features_df["total_products"].describe()

Valores nulos: 0


,total_accounts,total_cards,total_loans,total_products
0,1,4,0,5
1,1,1,0,2
2,1,1,1,3
3,2,2,0,4
4,2,1,1,4


count   50000.00
mean        4.10
std         3.28
min         0.00
25%         2.00
50%         4.00
75%         6.00
max        24.00
Name: total_products, dtype: float64

### Análisis e interpretación

La variable **total_products** se ha generado correctamente y no presenta valores nulos.

Los resultados muestran un promedio de **4,10 productos por cliente**, con una mediana de **4 productos**, lo que indica que la mayor parte de los clientes mantiene una relación relativamente consolidada con la entidad. Asimismo, se identifican clientes con hasta **24 productos financieros**, reflejando la existencia de perfiles con un elevado nivel de vinculación.

### Business Insight

El número total de productos constituye un indicador directo del grado de vinculación de cada cliente con la entidad.

Esta variable facilitará la identificación de clientes con alta vinculación comercial y servirá como base para futuras segmentaciones y análisis en el dashboard de Power BI.

## 5.2 Average Transaction Amount

### Descripción

Calcula el importe medio de las transacciones realizadas por cada cliente a partir del importe total y del número de transacciones registradas.

### Valor para el negocio

Esta variable permite diferenciar clientes con un mismo volumen total de transacciones pero con patrones de comportamiento distintos. Facilita la identificación de clientes que realizan muchas operaciones de pequeño importe frente a aquellos que realizan pocas operaciones de mayor valor.

In [34]:
# =============================================================================
# FV-002 · AVERAGE TRANSACTION AMOUNT
# =============================================================================

customer_features_df["avg_transaction_amount"] = np.where(
    customer_features_df["total_transactions"] > 0,
    customer_features_df["total_transaction_amount"]
    / customer_features_df["total_transactions"],
    0
)


In [35]:
# =============================================================================
# VERIFICACIÓN · AVERAGE TRANSACTION AMOUNT
# =============================================================================

print(f"Valores nulos: {customer_features_df['avg_transaction_amount'].isna().sum()}")

display(
    customer_features_df[
        [
            "total_transactions",
            "total_transaction_amount",
            "avg_transaction_amount"
        ]
    ].head()
)

customer_features_df[["avg_transaction_amount"]].describe().T

Valores nulos: 0


,total_transactions,total_transaction_amount,avg_transaction_amount
0,6,26890.44,4481.74
1,16,76937.17,4808.57
2,9,33025.48,3669.50
3,20,107590.15,5379.51
4,26,120446.58,4632.56


,count,mean,std,min,25%,50%,75%,max
avg_transaction_amount,50000.00,3882.69,2162.23,0.00,3755.75,4779.93,5279.51,8612.15


### Análisis e interpretación

La variable **avg_transaction_amount** se ha generado correctamente y no presenta valores nulos.

El importe medio por transacción es de **3.882,69 USD**, mientras que la mediana alcanza **4.779,93 USD**, lo que refleja diferencias en los patrones de uso entre clientes. Además, se observan importes medios de hasta **8.612,15 USD**, evidenciando la existencia de clientes que realizan operaciones de mayor cuantía.

### Business Insight

El importe medio por transacción aporta una visión complementaria al volumen total de operaciones realizadas por cada cliente.

Esta variable permitirá diferenciar perfiles con comportamientos transaccionales distintos y enriquecerá los análisis de actividad financiera en Power BI.

## 5.3 Average Loan Amount

### Descripción

Calcula el importe medio de los préstamos contratados por cada cliente a partir del importe total financiado y del número total de préstamos.

### Valor para el negocio

Esta variable permite identificar diferentes perfiles de financiación, diferenciando clientes con préstamos de elevada cuantía de aquellos con varios préstamos de menor importe. Complementa la información proporcionada por el importe total financiado y facilita el análisis del comportamiento crediticio en Power BI.

In [36]:
# =============================================================================
# FV-003 · AVERAGE LOAN AMOUNT
# =============================================================================

customer_features_df["avg_loan_amount"] = np.where(
    customer_features_df["total_loans"] > 0,
    customer_features_df["total_loan_amount"] /
    customer_features_df["total_loans"],
    0
)

In [37]:
# =============================================================================
# VERIFICACIÓN · AVERAGE LOAN AMOUNT
# =============================================================================

print(f"Valores nulos: {customer_features_df['avg_loan_amount'].isna().sum()}")

display(
    customer_features_df[
        [
            "total_loans",
            "total_loan_amount",
            "avg_loan_amount"
        ]
    ].head()
)

print(
    f"Clientes sin préstamos: "
    f"{(customer_features_df['total_loans'] == 0).sum():,}"
)

Valores nulos: 0


,total_loans,total_loan_amount,avg_loan_amount
0,0,0.00,0.00
1,0,0.00,0.00
2,1,204044.40,204044.40
3,0,0.00,0.00
4,1,61678.80,61678.80


Clientes sin préstamos: 27,460


### Análisis e interpretación

La variable **avg_loan_amount** se ha generado correctamente y no presenta valores nulos.

Del total de clientes analizados, **27.460 (54,92%) no tienen préstamos contratados**, por lo que el valor **0** representa una decisión de diseño adoptada para identificar a los clientes sin financiación.

Entre los clientes con préstamos activos se observan diferencias significativas en el importe medio financiado, alcanzando valores cercanos a los **300.000 USD**, lo que evidencia la existencia de perfiles de financiación muy diversos.

### Business Insight

El importe medio de los préstamos permite diferenciar clientes según su perfil de financiación, complementando la información aportada por el importe total financiado.

Esta variable facilitará la identificación de clientes con necesidades de financiación de distinta magnitud y enriquecerá los análisis de comportamiento crediticio en el dashboard de Power BI.

## 5.4 Engagement Level

### Descripción

Clasifica a los clientes según su nivel de vinculación con la entidad a partir del número total de productos financieros contratados.

### Valor para el negocio

Esta variable transforma una métrica numérica en una categoría de negocio fácilmente interpretable.

Permitirá segmentar clientes según su nivel de vinculación, facilitando el análisis comercial, la comparación entre grupos y el diseño de visualizaciones más intuitivas en Power BI.

### Decisión de diseño

El nivel de vinculación se ha definido mediante **reglas de negocio** en lugar de utilizar cuartiles o percentiles.

Aunque la distribución de la variable `total_products` se ha utilizado como referencia para establecer los umbrales, se ha optado por categorías estables y fácilmente interpretables por los usuarios de negocio.

Esta decisión facilita la comprensión de los resultados, simplifica el diseño del dashboard en Power BI y evita que los límites de cada categoría cambien si en el futuro se incorporan nuevos datos al modelo analítico.

In [38]:
# =============================================================================
# FV-004 · ENGAGEMENT LEVEL
# =============================================================================

conditions = [
    customer_features_df["total_products"] <= 2,
    customer_features_df["total_products"].between(3, 5),
    customer_features_df["total_products"] >= 6,
]

choices = [
    "Low",
    "Medium",
    "High",
]

customer_features_df["engagement_level"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

In [39]:
# =============================================================================
# VERIFICACIÓN · ENGAGEMENT LEVEL
# =============================================================================

print(f"Valores nulos: {customer_features_df['engagement_level'].isna().sum()}")

display(
    customer_features_df[
        [
            "total_products",
            "engagement_level"
        ]
    ].head(10)
)

customer_features_df["engagement_level"].value_counts()

Valores nulos: 0


,total_products,engagement_level
0,5,Medium
1,2,Low
2,3,Medium
3,4,Medium
4,4,Medium
5,1,Low
6,4,Medium
7,4,Medium
8,1,Low
9,1,Low


engagement_level
Low       18424
Medium    17253
High      14323
Name: count, dtype: int64

### Análisis e interpretación

La variable **engagement_level** se ha generado correctamente y no presenta valores nulos.

La distribución de clientes entre las tres categorías es equilibrada, con un **36,85 %** de clientes clasificados como **Low**, un **34,51 %** como **Medium** y un **28,65 %** como **High**.

Estos resultados indican que las reglas de negocio definidas permiten diferenciar adecuadamente distintos niveles de vinculación sin concentrar la mayoría de los clientes en una única categoría, facilitando su utilización en análisis comparativos y segmentaciones posteriores.

### Business Insight

La clasificación del nivel de vinculación transforma una métrica numérica en una categoría de negocio fácilmente interpretable.

Esta segmentación permitirá comparar el comportamiento financiero de clientes con distintos niveles de vinculación y facilitará la construcción de filtros, KPIs y visualizaciones orientadas al negocio en el dashboard de Power BI.

## 5.5 Multi Product Customer

### Descripción

Identifica si un cliente dispone de más de un producto financiero contratado.

### Valor para el negocio

Esta variable permite diferenciar rápidamente a los clientes multiproducto, facilitando análisis comerciales, segmentaciones y comparaciones entre clientes con distintos niveles de vinculación.

In [40]:
# =============================================================================
# FV-005 · MULTI PRODUCT CUSTOMER
# =============================================================================

customer_features_df["multi_product"] = np.where(
    customer_features_df["total_products"] > 1,
    "Yes",
    "No"
)

In [44]:
# =============================================================================
# DISTRIBUCIÓN DE MULTI PRODUCT
# =============================================================================

distribution = pd.DataFrame({
    "Clientes": customer_features_df["multi_product"].value_counts(),
    "Porcentaje (%)": round(
        customer_features_df["multi_product"]
        .value_counts(normalize=True)
        .mul(100),
        2
    )
})

distribution

,Clientes,Porcentaje (%)
multi_product,,
Yes,37835,75.67
No,12165,24.33


### Decisión de diseño

El nivel de saldo se ha definido mediante **reglas de negocio** en lugar de utilizar cuartiles o percentiles.

Aunque la distribución de la variable `total_balance` se ha utilizado como referencia para establecer los umbrales, se ha optado por categorías estables y fácilmente interpretables por los usuarios de negocio.

Esta decisión facilita la comprensión de los resultados, simplifica el diseño del dashboard en Power BI y evita que los límites de cada categoría cambien cuando el modelo se actualice con nuevos datos.

## 5.6 Balance Level

### Descripción

Clasifica a los clientes según el saldo total mantenido en sus cuentas bancarias.

### Valor para el negocio

Esta variable transforma el saldo total en una categoría de negocio fácilmente interpretable, permitiendo segmentar clientes según su patrimonio dentro de la entidad y facilitando el análisis comercial en Power BI.

### Decisión de diseño

El nivel de saldo se ha definido mediante **reglas de negocio** en lugar de utilizar directamente cuartiles o percentiles.

Aunque la distribución de la variable `total_balance` se ha utilizado como referencia para establecer los umbrales, se han seleccionado valores redondeados y fácilmente interpretables por los usuarios de negocio.

Los límites de **50.000 USD** y **200.000 USD** representan niveles de patrimonio sencillos de comunicar y próximos a la distribución observada en los datos, permitiendo identificar clientes con baja, media y alta capacidad financiera sin depender de umbrales estadísticos que podrían variar con futuras actualizaciones del dataset.

Esta decisión facilita la interpretación de los resultados, mejora la estabilidad del modelo analítico y simplifica el diseño del dashboard en Power BI.

In [46]:
# =============================================================================
# FV-006 · BALANCE LEVEL
# =============================================================================

conditions = [
    customer_features_df["total_balance"] < 50000,
    customer_features_df["total_balance"].between(50000, 200000),
    customer_features_df["total_balance"] > 200000
]

choices = [
    "Low",
    "Medium",
    "High"
]

customer_features_df["balance_level"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

In [47]:
# =============================================================================
# VERIFICACIÓN · BALANCE LEVEL
# =============================================================================

print(f"Valores nulos: {customer_features_df['balance_level'].isna().sum()}")

display(
    customer_features_df[
        [
            "total_balance",
            "balance_level"
        ]
    ].head(10)
)

distribution = pd.DataFrame({
    "Clientes": customer_features_df["balance_level"].value_counts(),
    "Porcentaje (%)": round(
        customer_features_df["balance_level"]
        .value_counts(normalize=True)
        .mul(100),
        2
    )
})

distribution

Valores nulos: 0


,total_balance,balance_level
0,151677.32,Medium
1,178960.28,Medium
2,131026.41,Medium
3,81689.89,Medium
4,189090.18,Medium
5,0.00,Low
6,86233.12,Medium
7,239077.17,High
8,170167.29,Medium
9,0.00,Low


,Clientes,Porcentaje (%)
balance_level,,
Medium,19487,38.97
Low,15847,31.69
High,14666,29.33


### Análisis e interpretación

La variable **balance_level** se ha generado correctamente y no presenta valores nulos.

La distribución de clientes entre las tres categorías es equilibrada, con **15.847 clientes (31,69%)** clasificados como **Low**, **19.487 clientes (38,97%)** como **Medium** y **14.666 clientes (29,33%)** como **High**.

Estos resultados indican que las reglas de negocio definidas permiten segmentar adecuadamente a los clientes según su capacidad financiera, manteniendo una distribución homogénea que facilitará el análisis comparativo y la construcción de visualizaciones en Power BI.

### Business Insight

La clasificación del saldo total en niveles de capacidad financiera simplifica el análisis patrimonial de la cartera de clientes.

Esta variable permitirá comparar el comportamiento de clientes con distinto nivel de patrimonio, facilitando la identificación de segmentos de mayor valor y apoyando el desarrollo de análisis comerciales y estratégicos en el dashboard de Power BI.

In [48]:
# =============================================================================
# VALIDACIÓN FINAL DEL DATASET
# =============================================================================

print(f"Filas: {customer_features_df.shape[0]}")
print(f"Columnas: {customer_features_df.shape[1]}")

customer_features_df.info()

customer_features_df.isna().sum()

Filas: 50000
Columnas: 15
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               50000 non-null  str    
 1   credit_score              50000 non-null  int64  
 2   total_accounts            50000 non-null  int64  
 3   total_balance             50000 non-null  float64
 4   total_cards               50000 non-null  int64  
 5   total_loans               50000 non-null  int64  
 6   total_loan_amount         50000 non-null  float64
 7   total_transactions        50000 non-null  int64  
 8   total_transaction_amount  50000 non-null  float64
 9   total_products            50000 non-null  int64  
 10  avg_transaction_amount    50000 non-null  float64
 11  avg_loan_amount           50000 non-null  float64
 12  engagement_level          50000 non-null  str    
 13  multi_product             50000 non-null  str 

customer_id                 0
credit_score                0
total_accounts              0
total_balance               0
total_cards                 0
total_loans                 0
total_loan_amount           0
total_transactions          0
total_transaction_amount    0
total_products              0
avg_transaction_amount      0
avg_loan_amount             0
engagement_level            0
multi_product               0
balance_level               0
dtype: int64

In [49]:
customer_features_df.head()

,customer_id,credit_score,total_accounts,total_balance,total_cards,total_loans,total_loan_amount,total_transactions,total_transaction_amount,total_products,avg_transaction_amount,avg_loan_amount,engagement_level,multi_product,balance_level
0,CUS000MKX5RHTAP,827,1,151677.32,4,0,0.00,6,26890.44,5,4481.74,0.00,Medium,Yes,Medium
1,CUS002V4AVJO5UQ,510,1,178960.28,1,0,0.00,16,76937.17,2,4808.57,0.00,Low,Yes,Medium
2,CUS004THQ8NDQW3,636,1,131026.41,1,1,204044.40,9,33025.48,3,3669.50,204044.40,Medium,Yes,Medium
3,CUS007GCM2J726A,492,2,81689.89,2,0,0.00,20,107590.15,4,5379.51,0.00,Medium,Yes,Medium
4,CUS00AO13A3Q5FO,686,2,189090.18,1,1,61678.80,26,120446.58,4,4632.56,61678.80,Medium,Yes,Medium


In [50]:
customer_features_df.duplicated(
    subset="customer_id"
).sum()

np.int64(0)

In [52]:
# =============================================================================
# EXPORTACIÓN DEL DATASET ENRIQUECIDO
# =============================================================================

OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "customer_features.csv"

try:
    customer_features_df.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print(f"✅ Dataset exportado correctamente en:\n{OUTPUT_PATH}")

except Exception as error:
    print(f"❌ Error al exportar el dataset: {error}")
    raise

✅ Dataset exportado correctamente en:
c:\Users\Lenovo\OneDrive\Documentos\Data_Projects\retail-banking-customer-intelligence\data\processed\customer_features.csv


# 7. Conclusiones

Durante este notebook se ha enriquecido el dataset analítico mediante la creación de seis variables derivadas orientadas al negocio.

Las nuevas variables incorporan información adicional sobre el nivel de vinculación, el comportamiento transaccional, el perfil de financiación y la capacidad financiera de los clientes, complementando el análisis realizado previamente en SQL y en el Notebook 02.

El dataset resultante presenta **50.000 registros**, **15 variables** y no contiene valores nulos en ninguna de sus columnas, quedando preparado para su utilización en el Notebook 04 y para el desarrollo del dashboard en Power BI.